In [2]:
import pandas as pd
df=pd.read_csv("updated_file.csv")

In [3]:
df.head()

,Query,Label,Sentence Length,AND Count,OR Count,UNION Count,Single Quote Count,Double Quote Count,Constant Value Count,Parentheses Count,Special Characters Total
0,""" or pg_sleep ( __TIME__ ) --",1,33,0,1,0,0,1,0,2,5
1,create user name identified by pass123 tempora...,1,90,0,0,0,0,0,0,0,1
2,AND 1 = utl_inaddr.get_host_address ( ...,1,217,2,0,0,3,0,2,10,6
3,select * from users where id = '1' or @ @1 ...,1,89,0,1,1,3,0,5,2,3
4,"select * from users where id = 1 or 1#"" ( ...",1,84,0,1,1,0,1,4,3,2


In [11]:
df['Label'].value_counts()

Label
0    19537
1    11382
Name: count, dtype: int64

In [12]:
print(df.isnull().sum())

Query                       0
Label                       0
Sentence Length             0
AND Count                   0
OR Count                    0
UNION Count                 0
Single Quote Count          0
Double Quote Count          0
Constant Value Count        0
Parentheses Count           0
Special Characters Total    0
dtype: int64


# Option 1: Oversampling (Increase Minority Class)

## Use SMOTE (Synthetic Minority Oversampling Technique) to generate synthetic malicious examples:

In [14]:
from imblearn.over_sampling import RandomOverSampler
import pandas as pd

# Assuming df is your dataset
X = df.drop('Label', axis=1)
y = df['Label']

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)

print(pd.Series(y_resampled).value_counts())

Label
1    19537
0    19537
Name: count, dtype: int64


### Step 3a: Text Preprocessing

#### We’ll clean the SQL queries so your ML model can learn patterns effectively.

Common steps:

Lowercase everything

Remove unnecessary whitespace

Optional: Remove comments (--, /* */)

Tokenization (split words/SQL keywords)

Optional: Remove stopwords (not always needed for SQL queries)

In [15]:
import re

def clean_query(query):
    query = query.lower()  # lowercase
    query = re.sub(r'--.*', '', query)  # remove single line comments
    query = re.sub(r'/\*.*?\*/', '', query, flags=re.DOTALL)  # remove block comments
    query = re.sub(r'\s+', ' ', query)  # normalize spaces
    query = query.strip()
    return query

df['clean_query'] = df['Query'].apply(clean_query)

In [18]:
df[['Query','clean_query']]

,Query,clean_query
0,""" or pg_sleep ( __TIME__ ) --",""" or pg_sleep ( __time__ )"
1,create user name identified by pass123 tempora...,create user name identified by pass123 tempora...
2,AND 1 = utl_inaddr.get_host_address ( ...,and 1 = utl_inaddr.get_host_address ( ( select...
3,select * from users where id = '1' or @ @1 ...,select * from users where id = '1' or @ @1 = 1...
4,"select * from users where id = 1 or 1#"" ( ...","select * from users where id = 1 or 1#"" ( unio..."
...,...,...
30914,DELETE FROM door WHERE grow = 'small',delete from door where grow = 'small'
30915,DELETE FROM tomorrow,delete from tomorrow
30916,SELECT wide ( s ) FROM west,select wide ( s ) from west
30917,SELECT * FROM ( SELECT slide FROM breath ),select * from ( select slide from breath )


#### Step 3b: TF-IDF Vectorization

Converts your cleaned SQL query text into numeric features for ML.

Captures word importance in queries.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
X_text = tfidf.fit_transform(df['clean_query'])

#### Step 3c: Combine TF-IDF + extra features

You already have features like:
AND Count, OR Count, UNION Count, Single Quote Count, Double Quote Count, Constant Value Count, Parentheses Count, Special Characters Total

In [20]:
import numpy as np
from scipy.sparse import hstack

# Extract extra numeric features
extra_features = df[['AND Count','OR Count','UNION Count','Single Quote Count',
                     'Double Quote Count','Constant Value Count','Parentheses Count',
                     'Special Characters Total']].values

# Combine with TF-IDF sparse matrix
X_final = hstack([X_text, extra_features])
y = df['Label'].values

### Step 4a: Choose ML Model

For SQL injection detection, common choices are:

#### Logistic Regression

Simple, fast, interpretable

Supports class_weight='balanced' for imbalanced data

#### Random Forest / Gradient Boosting

Handles numeric + sparse features well

Can capture complex patterns

#### XGBoost / LightGBM

Usually higher accuracy

Needs proper feature scaling & parameter tuning

For beginners, Logistic Regression or Random Forest is a good start.

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)

In [25]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [24]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [31]:
import xgboost as xgb
# Initialize XGBoost classifier
xgb_model =xgb.XGBClassifier(
    n_estimators=200,      # number of trees
    max_depth=6,           # depth of each tree
    learning_rate=0.1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # balance classes
    random_state=42,
    
    eval_metric='logloss'
)

# Train
xgb_model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [35]:
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
# Predict on test set
y_pred = model.predict(X_test)

# Evaluate
print("accuracy score:",round(accuracy_score(y_test,y_pred)*100))
print("Classification Report for Logistic Regression:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix for Logistic Regression::")
print(confusion_matrix(y_test, y_pred))

accuracy score: 99
Classification Report for Logistic Regression:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      3908
           1       1.00      0.99      0.99      2276

    accuracy                           0.99      6184
   macro avg       0.99      0.99      0.99      6184
weighted avg       0.99      0.99      0.99      6184

Confusion Matrix for Logistic Regression::
[[3897   11]
 [  34 2242]]


In [36]:
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
# Predict on test set
y_pred = rf_model.predict(X_test)

# Evaluate
print("accuracy score:",round(accuracy_score(y_test,y_pred)*100))
print("Classification Report for Logistic Regression:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix for Logistic Regression::")
print(confusion_matrix(y_test, y_pred))

accuracy score: 100
Classification Report for Logistic Regression:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3908
           1       1.00      0.99      1.00      2276

    accuracy                           1.00      6184
   macro avg       1.00      1.00      1.00      6184
weighted avg       1.00      1.00      1.00      6184

Confusion Matrix for Logistic Regression::
[[3905    3]
 [  19 2257]]


In [37]:
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
# Predict on test set
y_pred = xgb_model.predict(X_test)

# Evaluate
print("accuracy score:",round(accuracy_score(y_test,y_pred)*100))
print("Classification Report for Logistic Regression:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix for Logistic Regression::")
print(confusion_matrix(y_test, y_pred))

accuracy score: 99
Classification Report for Logistic Regression:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      3908
           1       1.00      0.99      0.99      2276

    accuracy                           0.99      6184
   macro avg       1.00      0.99      0.99      6184
weighted avg       0.99      0.99      0.99      6184

Confusion Matrix for Logistic Regression::
[[3900    8]
 [  23 2253]]


In [38]:
import numpy as np
import re

def predict_query(query, model, tfidf_vectorizer):
    """
    Predict whether a SQL query is normal or malicious (SQL Injection)
    
    Parameters:
        query (str): Input SQL query
        model: Trained ML model (RandomForest, XGBoost, etc.)
        tfidf_vectorizer: Fitted TF-IDF vectorizer
    
    Returns:
        str: "Normal" or "SQL Injection"
    """

    # ---- Step 1: Clean the query ----
    def clean_query(q):
        q = q.lower()
        q = re.sub(r'--.*', '', q)  # remove single line comments
        q = re.sub(r'/\*.*?\*/', '', q, flags=re.DOTALL)  # remove block comments
        q = re.sub(r'\s+', ' ', q)
        return q.strip()

    query_clean = clean_query(query)

    # ---- Step 2: TF-IDF vector ----
    X_text = tfidf_vectorizer.transform([query_clean])

    # ---- Step 3: Extract extra numeric features ----
    and_count = query.lower().count('and')
    or_count = query.lower().count('or')
    union_count = query.lower().count('union')
    single_quote_count = query.count("'")
    double_quote_count = query.count('"')
    constant_value_count = len(re.findall(r'\b\d+\b', query))
    parentheses_count = query.count('(') + query.count(')')
    special_char_total = len(re.findall(r'[;=#\-]', query))

    extra_features = np.array([[and_count, or_count, union_count, single_quote_count,
                                double_quote_count, constant_value_count,
                                parentheses_count, special_char_total]])

    # ---- Step 4: Combine features ----
    from scipy.sparse import hstack
    X_final = hstack([X_text, extra_features])

    # ---- Step 5: Predict ----
    pred = model.predict(X_final)[0]

    return "Normal" if pred == 0 else "SQL Injection"

In [40]:
query = "SELECT * FROM users WHERE id = '1' OR 1=1 --"
result = predict_query(query, model=rf_model, tfidf_vectorizer=tfidf)
print(result)
# Output: SQL Injection

SQL Injection


In [42]:
# Safe query example
safe_query = "SELECT name, email FROM users WHERE id = 10"
result = predict_query(safe_query, model=rf_model, tfidf_vectorizer=tfidf)
print(result)
# Output: Normal

Normal


In [44]:
import pickle

# Save the trained Random Forest model
with open("sql_rf_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

# Save the fitted TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Model and TF-IDF vectorizer saved successfully!")

Model and TF-IDF vectorizer saved successfully!
